# T5.2a Secuencias enlazadas

En este cuaderno estudiaremos en detalle cómo representar secuencias de datos mediante **nodos enlazados** en Python, y las operaciones fundamentales sobre ellas: creación, recorrido, búsqueda, inserción y eliminación.

## 1. Clase `Node`

Un **nodo** es la unidad elemental de una secuencia enlazada. Almacena:
- `data`: el dato contenido en el nodo. En Python, esto será un puntero o referencia al objeto en cuestión.
- `next`: una referencia (enlace) al siguiente nodo de la secuencia, o `None` si es el último.


Dado que Python es un lenguaje de **tipado dinámico**, nuestra clase `Node` será genérica: podrá almacenar datos de cualquier tipo.

In [1]:
class Node:
    """Nodo de una secuencia enlazada.
    
    Atributos:
        data: el dato almacenado en el nodo.
        next: referencia al siguiente nodo (None si es el último).
    """
    def __init__(self, data, next=None):
        self.data = data
        self.next = next
    
    def __repr__(self):
        return f"Node({self.data})"

Si declarasemos los atributos como privados, deberíamos definir una interfaz de métodos getter y setter / propiedades.

Dado que la clase `Node` es típicamente de uso interno, y en aras de una mayor simplicidad, hemos declarado los atributos como públicos.

### Construcción

La clase `Node` se puede instanciar de dos maneras:

**a) Nodo sin enlace**: `Node(data)`. Útil para crear el último nodo de la secuencia.

In [2]:
seq = Node(9)

print(f"seq.data = {seq.data}, seq.next = {seq.next}")

seq.data = 9, seq.next = None


<div align="center">
    <img src="fig/node_single.png" alt="Nodo aislado" style="max-width:350px"/>
</div>


**b) Nodo enlazado**: `Node(data, next)`. Para insertar el nodo en una posición interna de la secuencia. 

In [3]:
seq = Node(6, seq)

print(f"seq.data = {seq.data}, seq.next = {seq.next}")
print(f"seq.next.data = {seq.next.data}")  # el dato 9

seq.data = 6, seq.next = Node(9)
seq.next.data = 9


<div align="center">
    <img src="fig/node_chain.png" alt="Nodo enlazado" style="max-width:450px"/>
</div>

**Nota importante**: 

Los nodos no se "mueven" en memoria. Su posición en la secuencia viene dada por las referencias entre nodos. 

Por claridad en las figuras, representaremos los nodos alineados y en orden.

## 2. Creación y construcción de secuencias enlazadas

### Ejemplo 1: Construcción paso a paso por inserción en cabeza

Construimos la secuencia $\langle -2, 5, 10 \rangle$ insertando cada nuevo nodo al principio:

<div align="center">
    <img src="fig/seq_build_trace.png" alt="Traza de construcción" style="max-width:550px"/>
</div>

In [4]:
# Ejemplo 1: Construcción paso a paso
seq = None               # Paso 1: secuencia vacía
seq = Node(10)           # Paso 2: un solo nodo
seq = Node(5, seq)       # Paso 3: 5 → 10
seq = Node(-2, seq)      # Paso 4: -2 → 5 → 10

# Verificamos recorriendo la secuencia
aux = seq
while aux != None:
    print(aux.data, end=" -> ")
    aux = aux.next
print(aux)

-2 -> 5 -> 10 -> None


### Ejemplo 2: Convertir una lista Python en una secuencia enlazada

Dada una lista (array) de Python, construimos una secuencia enlazada con los mismos elementos y en el mismo orden. 

Para ello, **recorremos la lista en orden inverso**, insertando cada elemento en cabeza:

<div align="center">
    <img src="fig/list_to_linked_trace.png" alt="Traza de lista a enlazada" style="max-width:600px"/>
</div>

In [5]:
def list_to_linked(my_list):
    seq = None
    for i in range(len(my_list) - 1, -1, -1):
        seq = Node(my_list[i], seq)
    return seq

# Prueba
l = [8, 2, 6, 4]
seq = list_to_linked(l)

# Verificamos
aux = seq
while aux != None:
    print(aux.data, end=" -> ")
    aux = aux.next
print(aux)

8 -> 2 -> 6 -> 4 -> None


### Ejemplo 3: Inserción de nodos al final de la secuencia

Mediante un puntero `last` al último nodo, podemos insertar eficientemente al final:

<div align="center">
    <img src="fig/seq_insert_tail_trace.png" alt="Traza de inserción al final" style="max-width:600px"/>
</div>

In [6]:
# Ejemplo 3: Inserción al final con puntero 'last'
seq = None
last = None

# Primer nodo
seq = Node(10)
last = seq

# Siguiente nodo al final
last.next = Node(5)
last = last.next

# Otro nodo al final
last.next = Node(-2)
last = last.next

# Verificamos
aux = seq
while aux != None:
    print(aux.data, end=" -> ")
    aux = aux.next
print(aux)

10 -> 5 -> -2 -> None


## 3. Recorrido de secuencias enlazadas

El **recorrido** de una secuencia enlazada consiste en visitar todos sus nodos, realizando una cierta operación `tratar()` sobre cada dato.

El patrón de recorrido utiliza un puntero auxiliar `aux` que avanza por la secuencia:

```python
aux = seq
while aux != None:     # o simplemente: "while aux"
    tratar(aux.data)   # operación sobre el dato
    aux = aux.next     # avanzar al siguiente nodo
```

<div align="center">
    <img src="fig/seq_traversal.png" alt="Patrón de recorrido" style="max-width:600px"/>
</div>

> **Atención**: Si `aux` fuera `None`, el acceso a `aux.data` o `aux.next` lanzaría una excepción `AttributeError`.

### Ejemplo 4: Saturar valores a un máximo

Dada una secuencia enlazada de enteros, queremos que los valores mayores que `max_val` tomen el valor `max_val`:

In [7]:
def saturate(seq, max_val):
    """Satura los valores de la secuencia enlazada seq a un valor máximo.
    Los datos mayores que max_val se reemplazan por max_val.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia enlazada.
        max_val: valor máximo permitido.
    """
    aux = seq
    while aux:
        if aux.data > max_val:
            aux.data = max_val
        aux = aux.next

# Prueba: saturar {7, 2, 9, 4} con max_val = 5
lista = [7, 2, 9, 4]
max_val = 5

# primero convertimos lista a secuencia enlazada
seq = list_to_linked(lista) 

print("Antes:  ", end="");
aux = seq
while aux: 
    print(aux.data, end=" -> ")
    aux = aux.next
print(aux)

# saturamos la secuencia
saturate(seq, max_val)

print("Después:", end=" "); 
aux = seq
while aux: 
    print(aux.data, end=" -> ")
    aux = aux.next
print(aux)

Antes:  7 -> 2 -> 9 -> 4 -> None
Después: 5 -> 2 -> 5 -> 4 -> None


### Ejemplo 5: Representación como cadena

Función que devuelve una cadena representando la secuencia enlazada:

In [8]:
def linked_to_str(seq):
    """Devuelve una representación en cadena de la secuencia enlazada.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
    Retorna:
        str: representación de la secuencia, e.g. "8 -> 2 -> 6 -> 4 -> NULL"
    """
    ret = ""
    aux = seq
    while aux:
        ret += str(aux.data) + " -> "
        aux = aux.next
    ret += "NULL"
    return ret

# Prueba
seq = list_to_linked([8, 2, 6, 4])
cadena = linked_to_str(seq)

print(cadena)

8 -> 2 -> 6 -> 4 -> NULL


## 4. Búsqueda de elementos

### Búsqueda por valor

Al igual que en los arrays, una búsqueda se puede estructurar como un recorrido que **termina prematuramente** cuando se encuentra el elemento buscado `d`:

```python
aux = seq
while aux != None and aux.data != d:
    aux = aux.next
```

- Si al terminar el bucle, `aux != None`: 
   + **éxito** en la búsqueda.
- Si por contra, `aux == None`: 
   + **fracaso** (el dato no se ha encontrado).

<div align="center">
    <img src="fig/seq_search.png" alt="Búsqueda en secuencia enlazada" style="max-width:600px"/>
</div>

### Ejemplo 6: Cambiar el signo de la primera ocurrencia de un dato

In [9]:
def cambiar_signo(seq, d):
    """Cambia el signo de la primera ocurrencia de d en la secuencia.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        d: dato a buscar.
    """
    # Búsqueda del primer nodo cuyo dato sea d
    aux = seq
    while aux and aux.data != d:
        aux = aux.next
    
    # Si la búsqueda termina con éxito, se cambia el signo
    if aux:
        aux.data = -d

# Prueba con éxito
seq = list_to_linked([7, 1, 9, 4])
print("Secuencia original:  ", linked_to_str(seq))

cambiar_signo(seq, 1)
print("Cambio signo con d=1 (éxito):", linked_to_str(seq))

# Prueba con fracaso
cambiar_signo(seq, 25)
print("Cambio signo con d=25 (fracaso):", linked_to_str(seq)) 

Secuencia original:   7 -> 1 -> 9 -> 4 -> NULL
Cambio signo con d=1 (éxito): 7 -> -1 -> 9 -> 4 -> NULL
Cambio signo con d=25 (fracaso): 7 -> -1 -> 9 -> 4 -> NULL


### Ejemplo 7: Buscar la posición de un dato

In [10]:
def buscar(seq, d):
    """Devuelve la posición de la primera ocurrencia de d en la secuencia,
    o -1 si no se encuentra.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        d: dato a buscar.
    Retorna:
        int: posición o -1 si no encontrado.
    """
    aux = seq
    i = 0
    while aux and aux.data != d:
        aux = aux.next
        i += 1
    
    if aux:
        return i
    else:
        return -1

# Prueba
seq = list_to_linked([7, 8, 9, 1])
print(f"Secuencia: {linked_to_str(seq)}")
print(f"Posición de 9: {buscar(seq, 9)}")   # 2
print(f"Posición de 25: {buscar(seq, 25)}") # -1

Secuencia: 7 -> 8 -> 9 -> 1 -> NULL
Posición de 9: 2
Posición de 25: -1


### Ejemplo 8: Búsqueda por posición

En una secuencia enlazada, el acceso al $i$-ésimo nodo requiere **recorrer** la secuencia hasta alcanzar esa posición.

Se puede enfocar como una búsqueda: estamos buscando el i-ésimo nodo.

In [11]:
def buscar_por_posicion(seq, pos):
    """Devuelve el dato en la posición pos de la secuencia enlazada.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        pos (int): posición del dato a obtener (0-indexed).
    Retorna:
        El dato almacenado en la posición pos.
    Lanza:
        IndexError si la posición no existe.
    """
    aux = seq
    i = 0
    while aux and i < pos:
        aux = aux.next
        i += 1
    
    if aux:
        return aux.data
    else:
        raise IndexError(f"Posición {pos} inexistente")

# Prueba
seq = list_to_linked([7, 8, 9, 1])

print(f"Secuencia: {linked_to_str(seq)}")
print(f"Dato en posición 0: {buscar_por_posicion(seq, 0)}")
print(f"Dato en posición 2: {buscar_por_posicion(seq, 2)}")

try:
    buscar_por_posicion(seq, 10)
except IndexError as e:
    print(f"Error: {e}")

Secuencia: 7 -> 8 -> 9 -> 1 -> NULL
Dato en posición 0: 7
Dato en posición 2: 9
Error: Posición 10 inexistente


### Ejemplo 9: Búsqueda con condición compleja

Buscar la primera ocurrencia de un dato positivo que sea un cuadrado perfecto.

Si la condición a comprobar para el dato es relativamente compleja, no es conveniente escribirla en la guarda del bucle. Usaremos una variable booleana en su lugar y realizaremos su evaluación en el cuerpo del bucle.

In [12]:
import math

def buscar_cuadrado_perfecto(seq):
    """Busca la primera ocurrencia de un dato positivo que sea 
    un cuadrado perfecto en la secuencia enlazada.
    
    Retorna:
        El dato encontrado, o None si no existe.
    """
    aux = seq
    encontrado = False
    
    while aux and not encontrado:
        if aux.data > 0:
            sq = math.sqrt(aux.data)
            if sq == int(sq):
                encontrado = True
            else:
                aux = aux.next
        else:
            aux = aux.next
    
    if encontrado:
        return aux.data
    return None

# Prueba
seq = list_to_linked([3, -4, 16, 7, 25])
print(f"Secuencia: {linked_to_str(seq)}")

resultado = buscar_cuadrado_perfecto(seq)
print(f"Primer cuadrado perfecto: {resultado}")  # 16

Secuencia: 3 -> -4 -> 16 -> 7 -> 25 -> NULL
Primer cuadrado perfecto: 16


## 5. Inserción de elementos

La inserción de un nuevo elemento supone **crear** un nuevo nodo y **enlazarlo** en la posición deseada. Gracias a la disposición dispersa en memoria, la inserción se puede resolver **sin desplazar** los datos existentes.

A grandes rasgos, insertar implica, asumiendo que el dato se debería insertar en una posición `i` de la secuencia:

1. Buscar/alcanzar el nodo que ocupa la posición `i-1` inmediatamente anterior a la posición de inserción `i`.
2. Crear un nuevo nodo que encapsule el dato a insertar en la secuencia. Su enlace `next` será el nodo que ocupa actualmente la posición de inserción `i` (o `None` si se inserta en última posición).
3. Modificar el enlace del nodo en la posición `i-1` para que apunte al nuevo nodo creado.


En general, se distinguen dos casos:

### a) Inserción al principio de la secuencia (caso especial)

La posición de inserción viene dada directamente por `seq`:

```python
seq = Node(d, seq)
```

<div align="center">
    <img src="fig/seq_insert_head.png" alt="Inserción en cabeza" style="max-width:550px"/>
</div>

### b) Inserción en cualquier otra posición (caso general)

Se requiere tener acceso al nodo `prev` que precede al nuevo nodo insertado (esto implica realizar una búsqueda previa del nodo precedente).

El nuevo nodo se inserta entre `prev` y `prev.next`:

```python
prev.next = Node(d, prev.next)
```

<div align="center">
    <img src="fig/seq_insert_middle.png" alt="Inserción intermedia" style="max-width:650px"/>
</div>

### Ejemplo 10: Inserción por posición

> **Nota importante**: A diferencia de C/C++, en Python no existe paso de parámetros por referencia. 
> 
> Por tanto, cuando la inserción modifica la cabeza de la secuencia (`pos == 0`), debemos **retornar** el nuevo primer nodo.

Usaremos dos punteros auxiliares:
- `aux`: buscará y apuntará al nodo que ocupa la posición de inserción.
- `prev`: siempre apuntará al nodo anterior/previo a `aux`.

La inserción del nuevo nodo se realizará entre `prev` y `aux`.

> Nota: se puede implementar usando un solo puntero `prev`, pero dada la diferencia negligible en cuanto a costes, preferiremos la versión con dos punteros por ser más sencilla y legible.


In [21]:
def insertar(seq, d, pos):
    """Inserta el dato d en la posición pos de la secuencia enlazada.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia (o None).
        d: dato a insertar.
        pos (int): posición de inserción (0-indexed).
    Retorna:
        Node: primer nodo de la secuencia resultante.
    Lanza:
        IndexError si la posición no es válida.
    """
    if pos < 0:
        raise IndexError(f"Posición de inserción {pos} no válida")
    
    if pos == 0:  # a) al principio
        return Node(d, seq)
    
    # b) en cualquier otra posición
    prev = None
    aux = seq # inicializamos aux al primer nodo de la secuencia
    i = 0
    while aux and i < pos:
        prev = aux
        aux = aux.next
        i += 1
    
    if i == pos:  # posición válida 
        # aux apunta al nodo que está en la posición pos
        prev.next = Node(d, aux)
    else:
        raise IndexError(f"Posición de inserción {pos} no válida")
    
    # Siempre devolvemos el puntero al primer nodo, 
    # aunque no se haya modificado:
    return seq 


# Prueba: insertar en posición intermedia
seq = list_to_linked([8, 2, 4])
print("Secuencia original:  ", linked_to_str(seq))
seq = insertar(seq, 5, 2)
print("insertar(seq, 5, 2):", linked_to_str(seq))

# Prueba: insertar al principio
seq = insertar(seq, 99, 0)
print("insertar(seq, 99, 0):", linked_to_str(seq))

# Prueba: insertar al final
seq = insertar(seq, -1, 5)
print("insertar(seq, -1, 5):", linked_to_str(seq))

# Prueba: insertar en posición negativa
print("insertar(seq, 0, -1): ", end=" ")
try:
    seq = insertar(seq, 0, -1)
except IndexError as e:
    print(f"Error: {e}")

# Prueba: insertar en posición no válida
print("insertar(seq, 0, 7): ", end=" ")
try:
    seq = insertar(seq, 0, 7)
except IndexError as e:
    print(f"Error: {e}")

Secuencia original:   8 -> 2 -> 4 -> NULL
insertar(seq, 5, 2): 8 -> 2 -> 5 -> 4 -> NULL
insertar(seq, 99, 0): 99 -> 8 -> 2 -> 5 -> 4 -> NULL
insertar(seq, -1, 5): 99 -> 8 -> 2 -> 5 -> 4 -> -1 -> NULL
insertar(seq, 0, -1):  Error: Posición de inserción -1 no válida
insertar(seq, 0, 7):  Error: Posición de inserción 7 no válida


### Ejemplo 11: Inserción ordenada

Dada una secuencia enlazada cuyos datos están **ordenados de menor a mayor**, insertar un dato `d` manteniendo la ordenación.

La inserción se realiza al inicio de aquella subsecuencia que contenga a los elementos que son mayores o iguales que `d`.

Usaremos dos punteros auxiliares:
- `aux`: buscará apuntar al inicio de la subsecuencia cuyos elementos son mayores que `d`.
- `prev`: siempre apuntará al nodo anterior/previo a `aux`.

La inserción del nuevo nodo se realizará entre `prev` y `aux`.

Por ejemplo, si queremos insertar el dato `7` en la secuencia `[1,5,8,12]`:

<div align="center">
    <img src="fig/seq_insert_sorted_trace.png" alt="Traza de inserción ordenada" style="max-width:680px"/>
</div>

In [ ]:
def insertar_ordenado(seq, d):
    """Inserta d en una secuencia enlazada ordenada, manteniendo el orden.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia ordenada.
        d: dato a insertar.
    Retorna:
        Node: primer nodo de la secuencia resultante.
    """
    prev = None
    aux = seq
    
    # Búsqueda del primer dato >= d
    while aux and d > aux.data:
        prev = aux
        aux = aux.next
    
    # Inserción de d
    if prev is None:  # Caso a): insertar en cabeza
        seq = Node(d, seq)
    else:             # Caso b): insertar entre prev y aux
        prev.next = Node(d, aux)
    
    return seq

# Prueba
seq = list_to_linked([1, 5, 8, 12])
print("Secuencia original:  ", linked_to_str(seq))

seq = insertar_ordenado(seq, 7)
print("insertar_ordenado(seq, 7):", linked_to_str(seq))

seq = insertar_ordenado(seq, 0)
print("insertar_ordenado(seq, 0):", linked_to_str(seq))

seq = insertar_ordenado(seq, 20)
print("insertar_ordenado(seq, 20):", linked_to_str(seq))

## 6. Eliminación de elementos

La eliminación de un dato se resuelve **sin desplazar** los datos existentes, simplemente modificando enlaces (*bypass*).


A grandes rasgos, eliminar implica, asumiendo que el dato a eliminar se encuentra en una posición `i` de la secuencia:

1. Buscar/alcanzar el nodo que ocupa la posición `i-1` inmediatamente anterior al nodo que contiene el dato a eliminar.
2. Modificar el enlace `next` del nodo anterior en posición `i-1` para que apunte al nodo que está en la posición `i+1` (o `None` si el Nodo a eliminar era el último).
3. Liberar la memoria ocupada por el nodo eliminado.


Se distinguen dos casos:

### a) Eliminar al principio de la secuencia

```python
seq = seq.next   # bypass
```

<div align="center">
    <img src="fig/seq_delete_head.png" alt="Eliminación en cabeza" style="max-width:550px"/>
</div>

### b) Eliminar en cualquier otra posición

Se requiere acceso al nodo predecesor `prev`, mientras que `aux` apuntará al nodo que será _"bypass-eado"_:

```python
prev.next = aux.next   # bypass
```

<div align="center">
    <img src="fig/seq_delete_middle.png" alt="Eliminación intermedia" style="max-width:650px"/>
</div>


> **Nota**: En C++, tras el bypass, el nodo eliminado debe ser destruido con `delete` para liberar memoria. En Python, el garbage collector se encarga automáticamente.

### Ejemplo 12: Eliminar por posición

Debemos gestionar la validez de la posición proporcionada. Aquí sabemos a priori si eliminamos al principio o en cualquier otro punto de la secuencia.

In [ ]:
def eliminar_pos(seq, pos):
    """Elimina el nodo en la posición pos de la secuencia enlazada.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        pos (int): posición del nodo a eliminar (0-indexed).
    Retorna:
        Node: primer nodo de la secuencia modificada.
    Lanza:
        IndexError si la posición no es válida o si la secuencia está vacía.
    """
    if pos < 0: 
        raise IndexError(f"Posición {pos} no válida")
    
    if seq is None:
        raise IndexError("Secuencia vacía")
    
    if pos == 0:  # a) eliminar el primero
        return seq.next
    
    # b) buscar el nodo en la posición pos
    prev = None
    aux = seq
    i = 0
    while aux and i < pos:
        prev = aux
        aux = aux.next
        i += 1
    
    if aux:   # éxito en la búsqueda
        prev.next = aux.next  # bypass
    else:
        raise IndexError(f"Posición {pos} no válida")
    
    return seq

# Prueba
seq = list_to_linked([8, 2, 6])
print("Secuencia original:", linked_to_str(seq))

seq = eliminar_pos(seq, 2)
print("Eliminar pos. 2:", linked_to_str(seq))

seq = eliminar_pos(seq, 0)
print("Eliminar pos. 0:", linked_to_str(seq))

# Prueba: eliminar en posición negativa
print("Eliminar pos. -1: ", end=" ")
try:
    seq = eliminar_pos(seq, -1)
except IndexError as e:
    print(f"Error: {e}")

# Prueba: eliminar en posición inválida
print("Eliminar pos. 2: ", end=" ")
try:
    seq = eliminar_pos(seq, 2)
except IndexError as e:
    print(f"Error: {e}")

seq = eliminar_pos(seq, 0)
print("Eliminar pos. 0:", linked_to_str(seq))

# Prueba: eliminar en secuencia vacía
print("Eliminar pos. 2: ", end=" ")
try:
    seq = eliminar_pos(seq, 2)
except IndexError as e:
    print(f"Error: {e}")

### Ejemplo 13: Eliminar la primera ocurrencia de un dato

No sabemos si el dato está, y en caso de estarlo, en qué posición se encontrará. 

Por tanto, realizamos un recorrido estándar con punteros `prev` y `aux` hasta encontrarlo o agotar la secuencia. 

Si lo encontramos, entonces estudiaremos si estamos en el caso a) o b) de eliminación.

In [ ]:
def eliminar_dato(seq, d):
    """Elimina, si existe, la primera ocurrencia del dato d.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        d: dato a eliminar.
    Retorna:
        Node: primer nodo de la secuencia resultante.
    """
    prev = None
    aux = seq
    
    # Búsqueda de d
    while aux and aux.data != d:
        prev = aux
        aux = aux.next
    
    if aux:       # éxito en la búsqueda
        if prev is None:      # caso a) primer elemento
            seq = aux.next
        else:                 # caso b) resto
            prev.next = aux.next  # bypass
    
    return seq

# Prueba
seq = list_to_linked([7, 1, 9, 4])
print("Original:", linked_to_str(seq))

seq = eliminar_dato(seq, 9)
print("Eliminar 9:", linked_to_str(seq))

seq = eliminar_dato(seq, 7)
print("Eliminar 7:", linked_to_str(seq))

seq = eliminar_dato(seq, 25)
print("Eliminar 25 (fallo):", linked_to_str(seq))  # sin cambios

### Ejemplo 14: Eliminar todos los nodos menores que un valor mínimo

Aquí no sabemos cuantos nodos eliminaremos, por lo que debemos recorrer toda la secuencia y eliminar cuando pertoque. 

Notar que cuando eliminamos un nodo apuntado por `aux`, `prev` no se modifica. 

In [ ]:
def eliminar_menores_que(seq, min_val):
    """Elimina todos los nodos cuyo dato sea menor que min_val.
    
    Parámetros:
        seq (Node): primer nodo de la secuencia.
        min_val: valor mínimo; se eliminan los datos estrictamente menores.
    Retorna:
        Node: primer nodo de la secuencia resultante.
    """
    prev = None
    aux = seq
    
    while aux:
        if aux.data < min_val:  # eliminar este nodo
            if prev is None:    # a) es el primero
                seq = aux.next
            else:               # b) resto de casos
                prev.next = aux.next  # bypass
            # No actualizamos prev (sigue siendo el mismo)
            aux = aux.next
        else:                   # no eliminar, avanzar ambos punteros
            prev = aux
            aux = aux.next
    
    return seq

# Prueba: eliminar menores que 10 en la secuencia {7, 15, 2, 1, 11, 0}
seq = list_to_linked([7, 15, 2, 1, 11, 0])
print("Secuencia original:", linked_to_str(seq))

seq = eliminar_menores_que(seq, 10)
print("Eliminar menores que 10:", linked_to_str(seq))